In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
import os

In [ ]:
# 设置绘图样式
plt.rcParams.update({
    'font.size': 24,  
    'font.weight': 'bold', 
    'axes.titlesize': 32, 
    'axes.labelsize': 28,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 28,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
import seaborn as sns
from matplotlib import cm

# ================= 参数设置 =================
T = 4          # 终止时间
dt = 0.01           # 时间步长
N = int(T / dt)     # 时间步数量
n_simulations = 10000  # 模拟路径数量

# SDE参数
pi = 100             # pi常数

beta = 0.2         # 参数beta
alpha = 0.5        # 参数alpha

SAVE_DIR = "./generated/one-period-result-fig"
# 初始化路径数组
A_paths = np.zeros((n_simulations, N+1))

# 初始条件
A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi

# 时间序列
time_grid = np.linspace(0, T, N+1)

# 模拟路径
for i in range(N):
    t = i * dt
    sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
    
    # 确保A值在[0, pi]范围内，避免数值问题
    A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
    
    # \Phi 和 \Phi^{-1} 的计算
    Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
    Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

    # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
    dW = np.sqrt(dt) * np.random.randn(n_simulations)
    dA = pi * Phi_prime * sqrt_zt * dW
    
    # 更新A值并确保在[0, pi]范围内
    A_next = A_current + dA
    A_paths[:, i+1] = np.clip(A_next, 0, pi)

# 分析结果：计算终点的分布
A_T = A_paths[:, -1]

# 创建保存图表的目录
plots_dir = os.path.join(SAVE_DIR, "carbon-dynamic")
os.makedirs(plots_dir, exist_ok=True)

# 选取多个时间节点来绘制KDE
time_points = np.linspace(0, N, 11, dtype=int)  # 选取10个时间点
colors = cm.viridis(np.linspace(0, 1, len(time_points)))[::-1]  # 生成从深到浅的颜色渐变

# =================================================================
# 修复版本：正确收集图例句柄
# =================================================================

# 方法1：使用显式收集图例句柄
plt.figure(figsize=(28, 8))
ax1 = plt.subplot(1, 2, 1)

# 创建图例的句柄和标签
handles = []
labels = []

for i, (t, color) in enumerate(zip(time_points, colors)):
    # 显式获取线条对象
    sns_line = sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, 
                          label=f'Time = {time_grid[t]:.2f}', ax=ax1)
    
    # 获取最后添加的线对象（更可靠的方法）
    if ax1.lines:  # 确保有线条存在
        line = ax1.lines[-1]
        handles.append(line)
        labels.append(f'Time = {time_grid[t]:.2f}')

plt.xlabel('A(t)')
plt.ylabel('Density')
plt.title('Time-evolving KDE of $A_t$')

# 添加边界线
plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Lower bound (0)')
plt.axvline(x=pi, color='red', linestyle='--', alpha=0.7, label='Upper bound (π)')

# 方法2：使用get_legend_handles_labels（更简单）
if not handles:  # 如果方法1失败，使用方法2
    handles, labels = ax1.get_legend_handles_labels()

# 添加边界线到图例
boundary_handles = [
    plt.Line2D([0], [0], color='red', linestyle='--', alpha=0.7),
    plt.Line2D([0], [0], color='red', linestyle='--', alpha=0.7)
]
boundary_labels = ['Lower bound (0)', 'Upper bound (π)']

if handles:  # 确保我们有图例句柄
    # 将图例分成两部分：一部分在左侧，一部分在右侧
    split_index = len(handles) // 2  # 将图例分成两半

    # 左侧图例
    left_legend = plt.legend(handles[:split_index], labels[:split_index], 
                             loc='upper left', bbox_to_anchor=(0.01, 0.98),
                             frameon=True, framealpha=0.9)

    # 右侧图例（添加到轴中但不覆盖左侧图例）
    plt.gca().add_artist(left_legend)  # 先添加左侧图例
    
    # 添加边界线图例
    plt.legend(boundary_handles, boundary_labels, 
               loc='upper right', bbox_to_anchor=(0.99, 0.98),
               frameon=True, framealpha=0.9)
else:
    print("Warning: No legend handles found. Adding standard legend.")
    plt.legend(loc='best')

# =================================================================
# 第二个子图：终点的直方图和核密度估计
# =================================================================
ax2 = plt.subplot(1, 2, 2)
plt.hist(A_T, bins=50, density=True, alpha=0.5, color='blue', label='Histogram of $A_T$')
sns.kdeplot(A_T, color='red', label='KDE of $A_T$', ax=ax2)

# 添加边界线
plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Lower bound (0)')
plt.axvline(x=pi, color='red', linestyle='--', alpha=0.7, label='Upper bound (π)')

plt.xlabel('$A_T$')
plt.ylabel('Density')
plt.title(f'Distribution of $A_T$')
plt.legend()

# 调整布局并保存
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "Dynamics of Carbon Allowance Prices.png"), dpi=300, bbox_inches='tight')
plt.show()

# =================================================================
# 额外分析：检查边界触碰情况
# =================================================================
print("Boundary analysis:")
print(f"Minimum value: {np.min(A_paths):.6f}")
print(f"Maximum value: {np.max(A_paths):.6f}")

# 检查是否有路径触碰边界
touched_lower = np.any(A_paths <= 0, axis=1)
touched_upper = np.any(A_paths >= pi, axis=1)

print(f"Paths that touched lower bound (0): {np.sum(touched_lower)}/{n_simulations} ({np.mean(touched_lower)*100:.2f}%)")
print(f"Paths that touched upper bound (π): {np.sum(touched_upper)}/{n_simulations} ({np.mean(touched_upper)*100:.2f}%)")

# # 绘制边界触碰统计
# plt.figure(figsize=(10, 6))
# categories = ['Lower Bound (0)', 'Upper Bound (π)']
# percentages = [np.mean(touched_lower)*100, np.mean(touched_upper)*100]

# bars = plt.bar(categories, percentages, color=['lightcoral', 'lightblue'])
# plt.ylabel('Percentage of Paths (%)')
# plt.title('Boundary Touching Statistics')

# # 在柱状图上添加数值标签
# for bar, percentage in zip(bars, percentages):
#     height = bar.get_height()
#     plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
#             f'{percentage:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "Boundary_Touching_Statistics.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
import seaborn as sns
from matplotlib import cm
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
# ================= 参数设置 =================
T = 4          # 终止时间
dt = 0.01           # 时间步长
N = int(T / dt)     # 时间步数量
n_simulations = 10000  # 模拟路径数量

# SDE参数
pi = 100             # pi常数

# 参数组合列表
param_combinations = [
    {'alpha': 1.2, 'beta': 0.8, 'label': 'α=1.2, β=0.8'},
    {'alpha': 0.8, 'beta': 0.5, 'label': 'α=0.8, β=0.5'},
    {'alpha': 0.4, 'beta': 0.2, 'label': 'α=0.4, β=0.2'}
]

# 创建保存图表的目录
SAVE_DIR = "./generated/one-period-result-fig"
plots_dir = os.path.join(SAVE_DIR, "carbon-dynamic")
os.makedirs(plots_dir, exist_ok=True)

# 时间序列
time_grid = np.linspace(0, T, N+1)

# 创建1×3的子图
fig, axes = plt.subplots(1, 3, figsize=(36, 8))
# 初始化存储每个子图最大y值的列表
subplot_max_density = [0, 0, 0]

# 创建全局图例的句柄和标签（用于时间点）
global_time_handles = []
global_time_labels = []
# 颜色设置
colors = cm.viridis(np.linspace(0, 1, 11))[::-1]  # 生成从深到浅的颜色渐变

# 对每个参数组合进行模拟
for idx, params in enumerate(param_combinations):
    alpha = params['alpha']
    beta = params['beta']
    label = params['label']
    
    # 初始化路径数组
    A_paths = np.zeros((n_simulations, N+1))
    
    # 初始条件
    A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    
    # 模拟路径
    for i in range(N):
        t = i * dt
        sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
        # 确保A值在[0, pi]范围内，避免数值问题
        A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
        # \Phi 和 \Phi^{-1} 的计算
        Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
        Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

        # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
        dW = np.sqrt(dt) * np.random.randn(n_simulations)
        dA = pi * Phi_prime * sqrt_zt * dW
        
        # 更新A值并确保在[0, pi]范围内
        A_next = A_current + dA
        A_paths[:, i+1] = np.clip(A_next, 0, pi)

    

    # 选取多个时间节点来绘制KDE
    time_points = np.linspace(0, N, 11, dtype=int)  # 选取10个时间点
    
    ax = axes[idx]
    
    # 绘制KDE并记录最大密度值
    current_max = 0
    for i, (t, color) in enumerate(zip(time_points, colors)):
        # 只对第一个参数组合添加标签（避免重复）
        if idx == 0:
            # label = f't={time_grid[t]:.2f}'
            sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, label=label, ax=ax)
        else:
            sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, ax=ax)
    # for i, (t, color) in enumerate(zip(time_points, colors)):
    #     kde_data = A_paths[:, t]
    #     # 计算KDE
    #     kde = stats.gaussian_kde(kde_data, bw_method=0.5)
    #     x_vals = np.linspace(0, pi, 1000)
    #     y_vals = kde(x_vals)
        
    #     # 更新当前最大密度值
    #     current_max = max(current_max, np.max(y_vals))
        
    #     # 只对第一个参数组合添加标签（避免重复）
    #     if idx == 0:
    #         ax.plot(x_vals, y_vals, color=color, label=f't={time_grid[t]:.2f}')
    #     else:
    #         ax.plot(x_vals, y_vals, color=color)
    
    # 存储当前子图的最大密度值（增加10%的余量）
    for line in ax.lines:
        y_data = line.get_ydata()
        if len(y_data) > 0:
            current_max = max(current_max, np.max(y_data))
    subplot_max_density[idx] = current_max * 1.1  # 增加10%的余量
    subplot_max_density[idx] = current_max * 1.1
    
    ax.set_xlabel('A(t)')
    ax.set_ylabel('Density')
    ax.set_title(f'Time-evolving KDE of $A_t$ ({label})')
    # ax.set_xlim(0, pi)  # 设置统一的X轴范围

# 计算所有子图的最大密度值
global_max_density = max(subplot_max_density)

# 设置统一y轴范围（从0开始）
for ax in axes:
    ax.set_ylim(0, global_max_density)

# 从第一个子图中获取图例句柄（用于全局图例）
if axes[0].lines:
    global_time_handles = [mlines.Line2D([0], [0], color=colors[i], lw=2) 
                          for i in range(len(time_points))]
    global_time_labels = [f't={time_grid[t]:.2f}' for t in time_points]

# 添加全局图例（在整个图形底部中央）
fig.legend(global_time_handles, global_time_labels, 
           loc='lower center', 
           bbox_to_anchor=(0.5, -0.08),
           ncol=11,  # 一行显示6个图例项（11个时间点分两行）
           frameon=True, 
           framealpha=0.9)

# 调整布局，为图例留出空间
plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # 底部留出更多空间给图例

plt.savefig(os.path.join(plots_dir, "Multi_Param_Dynamics_of_Carbon_Allowance_Prices.png"), 
           dpi=300, bbox_inches='tight')
plt.show()

# =================================================================
# 第二个图窗：终点的直方图和核密度估计
# =================================================================
fig2, axes2 = plt.subplots(1, 3, figsize=(36, 8))

# 创建全局图例的句柄和标签
global_handles = []
global_labels = []

# 对每个参数组合进行模拟（再次运行，或者可以保存A_T结果）
for idx, params in enumerate(param_combinations):
    alpha = params['alpha']
    beta = params['beta']
    label = params['label']
    
    # 初始化路径数组
    A_paths = np.zeros((n_simulations, N+1))
    
    # 初始条件
    A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    

    # 模拟路径
    for i in range(N):
        t = i * dt
        sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
        # 确保A值在[0, pi]范围内，避免数值问题
        A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
        # \Phi 和 \Phi^{-1} 的计算
        Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
        Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

        # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
        dW = np.sqrt(dt) * np.random.randn(n_simulations)
        dA = pi * Phi_prime * sqrt_zt * dW
        
        # 更新A值并确保在[0, pi]范围内
        A_next = A_current + dA
        A_paths[:, i+1] = np.clip(A_next, 0, pi)

    # 分析结果：计算终点的分布
    A_T = A_paths[:, -1]
    
    # 在当前子图中绘制
    ax = axes2[idx]
    
    # 绘制直方图和KDE
    hist = ax.hist(A_T, bins=50, density=True, alpha=0.5, color='blue', label='Histogram of $A_T$')
    kde = sns.kdeplot(A_T, color='red', label='KDE of $A_T$', ax=ax)
    
    # 保存图例句柄和标签（只保存一次）
    if idx == 0:
        # 获取直方图和KDE的句柄
        hist_handle = plt.Rectangle((0,0), 1, 1, fc='blue', alpha=0.5)
        kde_handle = plt.Line2D([0,1], [0,1], color='red')
        
        global_handles = [hist_handle, kde_handle]
        global_labels = ['Histogram of $A_T$', 'KDE of $A_T$']
    
    ax.set_xlabel('$A_T$')
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of $A_T$ ({label})')

# 添加全局图例（只在图片正下方添加一个图例）
fig2.legend(global_handles, global_labels, 
           loc='lower center', 
           bbox_to_anchor=(0.5, -0.05),
           ncol=2,  # 一行显示两个图例项
           frameon=True, 
           framealpha=0.9)

# 调整布局，为图例留出空间
plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # 底部留出5%的空间给图例

# 保存图片
plt.savefig(os.path.join(plots_dir, "Multi_Param_Distribution_of_AT.png"), 
           dpi=300, bbox_inches='tight')
plt.show()


n_paths_to_plot = 200

fig3, axes3 = plt.subplots(1, 3, figsize=(36, 8))

# 对每个参数组合进行模拟
for idx, params in enumerate(param_combinations):
    alpha = params['alpha']
    beta = params['beta']
    label = params['label']
    
    # 初始化路径数组
    A_paths = np.zeros((n_simulations, N+1))
    
    # 初始条件
    A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    
    # 模拟路径
    for i in range(N):
        t = i * dt
        sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
        # 确保A值在[0, pi]范围内，避免数值问题
        A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
        # \Phi 和 \Phi^{-1} 的计算
        Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
        Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

        # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
        dW = np.sqrt(dt) * np.random.randn(n_simulations)
        dA = pi * Phi_prime * sqrt_zt * dW
        
        # 更新A值并确保在[0, pi]范围内
        A_next = A_current + dA
        A_paths[:, i+1] = np.clip(A_next, 0, pi)

    # 在当前子图中绘制前n_paths_to_plot条路径
    ax = axes3[idx]
    
    # 使用浅色和透明度使图表更清晰
    for i in range(min(n_paths_to_plot, n_simulations)):
        ax.plot(time_grid, A_paths[i, :], alpha=0.1, color='blue')
    
    # 添加平均路径
    mean_path = np.mean(A_paths[:n_paths_to_plot, :], axis=0)
    mean_line, = ax.plot(time_grid, mean_path, color='red', linewidth=2)
    
    # 保存图例句柄（只保存一次）
    if idx == 0:
        # 创建模拟路径的图例句柄（使用透明蓝色矩形）
        # path_handle = plt.Rectangle((0,0), 1, 1, fc='blue', alpha=0.1)
        # global_path_handles = [path_handle, mean_line]
        # global_path_labels = ['Simulated Paths', 'Mean Path']
        global_path_handles = [ mean_line]
        global_path_labels = [ 'Mean Path']
    
    ax.set_xlabel('Time (t)')
    ax.set_ylabel('A(t)')
    ax.set_title(f'Simulated {n_paths_to_plot} Paths of $A_t$ ({label})')
    # ax.set_ylim(0, pi)

# 添加全局图例（在整个图形底部中央）
fig3.legend(global_path_handles, global_path_labels, 
           loc='lower center', 
           bbox_to_anchor=(0.5, -0.05),
           ncol=2,  # 一行显示两个图例项
           frameon=True, 
           framealpha=0.9)

# 调整布局，为图例留出空间
plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # 底部留出5%的空间给图例


# 保存图片
plt.savefig(os.path.join(plots_dir, "Multi_Param_Simulated_Paths.png"), 
           dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ================= 参数设置 =================
T = 4          # 终止时间
dt = 0.01           # 时间步长
N = int(T / dt)     # 时间步数量
n_simulations = 10000  # 模拟路径数量

# SDE参数
pi = 100             # pi常数

# 参数组合列表
param_combinations = [
    {'alpha': 1.2, 'beta': 0.8, 'label': 'α=1.2, β=0.8'},
    {'alpha': 0.8, 'beta': 0.5, 'label': 'α=0.8, β=0.5'},
    {'alpha': 0.4, 'beta': 0.2, 'label': 'α=0.4, β=0.2'}
]

# 创建保存图表的目录
SAVE_DIR = "./generated/one-period-result-fig"
plots_dir = os.path.join(SAVE_DIR, "carbon-dynamic")
os.makedirs(plots_dir, exist_ok=True)

# 时间序列
time_grid = np.linspace(0, T, N+1)

# =================================================================
# 新的合并图：第一行是KDE，第二行是模拟路径
# =================================================================
fig_combined, axes_combined = plt.subplots(2, 3, figsize=(36, 18))

# 调整子图之间的间距


# 初始化存储每个子图最大y值的列表（用于KDE图）
subplot_max_density = [0, 0, 0]

# 颜色设置
colors = cm.viridis(np.linspace(0, 1, 11))[::-1]  # 生成从深到浅的颜色渐变

# 对每个参数组合进行模拟
for idx, params in enumerate(param_combinations):
    alpha = params['alpha']
    beta = params['beta']
    label = params['label']
    
    # 初始化路径数组
    A_paths = np.zeros((n_simulations, N+1))
    
    # 初始条件
    A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    
    # 模拟路径
    for i in range(N):
        t = i * dt
        sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
        # 确保A值在[0, pi]范围内，避免数值问题
        A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
        # \Phi 和 \Phi^{-1} 的计算
        Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
        Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

        # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
        dW = np.sqrt(dt) * np.random.randn(n_simulations)
        dA = pi * Phi_prime * sqrt_zt * dW
        
        # 更新A值并确保在[0, pi]范围内
        A_next = A_current + dA
        A_paths[:, i+1] = np.clip(A_next, 0, pi)

    # ================= 第一行：KDE图 =================
    ax_kde = axes_combined[0, idx]
    
    # 选取多个时间节点来绘制KDE
    time_points = np.linspace(0, N, 11, dtype=int)  # 选取10个时间点
    
    # 绘制KDE并记录最大密度值
    current_max = 0
    for i, (t, color) in enumerate(zip(time_points, colors)):
        # 只对第一个参数组合添加标签（避免重复）
        if idx == 0:
            sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, label=f't={time_grid[t]:.2f}', ax=ax_kde)
        else:
            sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, ax=ax_kde)
    
    # 存储当前子图的最大密度值（增加10%的余量）
    for line in ax_kde.lines:
        y_data = line.get_ydata()
        if len(y_data) > 0:
            current_max = max(current_max, np.max(y_data))
    subplot_max_density[idx] = current_max * 1.1  # 增加10%的余量
    
    ax_kde.set_xlabel('A(t)')
    ax_kde.set_ylabel('Density')
    ax_kde.set_title(f'Time-evolving KDE of $A_t$ ({label})')
    ax_kde.text(0.02, 0.98, f'({chr(97+idx)})', transform=ax_kde.transAxes, 
                fontsize=24, fontweight='bold', va='top', ha='left'
                # bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8)
                )
    
    # ================= 第二行：模拟路径图 =================
    ax_paths = axes_combined[1, idx]
    n_paths_to_plot = 200
    
    # 使用浅色和透明度使图表更清晰
    for i in range(min(n_paths_to_plot, n_simulations)):
        ax_paths.plot(time_grid, A_paths[i, :], alpha=0.1, color='blue')
    
    # 添加平均路径
    # mean_path = np.mean(A_paths[:n_paths_to_plot, :], axis=0)
    # mean_line, = ax_paths.plot(time_grid, mean_path, color='red', linewidth=2)
    
    ax_paths.set_xlabel('Time (t)')
    ax_paths.set_ylabel('A(t)')
    ax_paths.set_title(f'Simulated {n_paths_to_plot} Paths of $A_t$ ({label})')
    ax_paths.text(0.02, 0.98, f'({chr(100+idx)})', transform=ax_paths.transAxes, 
                  fontsize=24, fontweight='bold', va='top', ha='left'
                #   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8)
                  )

# 计算所有KDE子图的最大密度值并设置统一y轴范围
global_max_density = max(subplot_max_density)
for idx in range(3):
    axes_combined[0, idx].set_ylim(0, global_max_density)

# 创建图例
# 时间点图例（从第一个子图获取）
if axes_combined[0, 0].lines:
    time_handles = [mlines.Line2D([0], [0], color=colors[i], lw=2) 
                   for i in range(len(time_points))]
    time_labels = [f't={time_grid[t]:.2f}' for t in time_points]
    
    # 路径图例
    path_handle = mlines.Line2D([0], [0], color='blue', linewidth=1)
    path_labels = ['$A_t$ Path']

# 添加图例
# 第一行图例（时间点）- 放在第一行下面
fig_combined.legend(time_handles, time_labels, 
                   loc='upper center', 
                   bbox_to_anchor=(0.5, 0.53),  # 调整位置到第一行下面
                   ncol=11,  # 调整为6列，使图例更紧凑
                   frameon=True, 
                   framealpha=0.9)

# 第二行图例（路径）- 放在第二行下面
fig_combined.legend([path_handle], path_labels,
                   loc='lower center',
                   bbox_to_anchor=(0.5, 0.02),
                   ncol=2,
                   frameon=True,
                   framealpha=0.9)

# 调整布局
plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # 为底部图例留出空间
plt.subplots_adjust(hspace=0.5)  # 增加行间距
# 保存合并图
plt.savefig(os.path.join(plots_dir, "Combined_KDE_and_Paths.png"), 
           dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
import seaborn as sns
from matplotlib import cm
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
plt.rcParams.update({
    'font.size': 48,  
    'font.weight': 'bold', 
    'axes.titlesize': 48, 
    'axes.labelsize': 48,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 28,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})

# ================= 参数设置 =================
T = 4          # 终止时间
dt = 0.01           # 时间步长
N = int(T / dt)     # 时间步数量
n_simulations = 10000  # 模拟路径数量

# SDE参数
pi = 100             # pi常数

# 只保留alpha=1.2, beta=0.8的参数组合
alpha = 1.2
beta = 0.8

# 选择要绘制的时间点（可以修改这个值来选择不同的时间点）
selected_time = 4.0 # 例如选择t=2.0
selected_time_index = int(selected_time / dt)  # 计算对应的时间索引

# 创建保存图表的目录
SAVE_DIR = "./generated/one-period-result-fig"
plots_dir = os.path.join(SAVE_DIR, "carbon-dynamic")
os.makedirs(plots_dir, exist_ok=True)

# 时间序列
time_grid = np.linspace(0, T, N+1)

# 创建单个图形
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# 初始化路径数组
A_paths = np.zeros((n_simulations, N+1))

# 初始条件
A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi

# 模拟路径
for i in range(N):
    t = i * dt
    sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
    
    # 确保A值在[0, pi]范围内，避免数值问题
    A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
    
    # \Phi 和 \Phi^{-1} 的计算
    Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
    Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

    # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
    dW = np.sqrt(dt) * np.random.randn(n_simulations)
    dA = pi * Phi_prime * sqrt_zt * dW
    
    # 更新A值并确保在[0, pi]范围内
    A_next = A_current + dA
    A_paths[:, i+1] = np.clip(A_next, 0, pi)

# 绘制选定时间点的KDE
sns.kdeplot(A_paths[:, selected_time_index], bw_adjust=0.5, color='black', ax=ax, linewidth=5)

# 添加统计信息
mean_val = np.mean(A_paths[:, selected_time_index])
std_val = np.std(A_paths[:, selected_time_index])

# 在图上添加统计信息文本框
stats_text = f'Mean: {mean_val:.3f}\nStd: {std_val:.3f}'
# ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=12,
#         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('A(t)')
ax.set_ylabel('Density')
# ax.set_title(f'KDE of $A_t$ at t={selected_time} (α={alpha}, β={beta})')
ax.set_xlim(-0.1*pi, 1.1*pi)
ax.set_ylim(0, 0.065)
ax.set_yticks([])
ax.grid(True, alpha=0.3)

# 移除网格
ax.grid(False)

# 移除上边框和右边框
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 移除y轴刻度
ax.set_yticks([])

# 设置x轴刻度 - 只显示0和π
ax.set_xticks([0, pi])
ax.set_xticklabels(['0', 'π'])



plt.tight_layout()
plt.savefig(os.path.join(plots_dir, f"Single_Time_KDE_alpha{alpha}_beta{beta}_t{selected_time}.png"), 
           dpi=800, bbox_inches='tight')
plt.show()

# 可选：打印一些额外的统计信息
print(f"Time: t = {selected_time}")
print(f"Mean: {mean_val:.4f}")
print(f"Standard Deviation: {std_val:.4f}")
print(f"Minimum: {np.min(A_paths[:, selected_time_index]):.4f}")
print(f"Maximum: {np.max(A_paths[:, selected_time_index]):.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
import seaborn as sns
from matplotlib import cm
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from mpl_toolkits.mplot3d import Axes3D  # 导入3D绘图工具

# ================= 参数设置 =================
T = 4          # 终止时间
dt = 0.01           # 时间步长
N = int(T / dt)     # 时间步数量
n_simulations = 10000  # 模拟路径数量

# SDE参数
pi = 100             # pi常数

# 参数组合列表 - 只保留一个参数组合
param_combinations = [
    {'alpha': 1.2, 'beta': 0.8, 'label': 'α=1.2, β=0.8'}
]

# 创建保存图表的目录
SAVE_DIR = "./generated/one-period-result-fig"
plots_dir = os.path.join(SAVE_DIR, "carbon-dynamic")
os.makedirs(plots_dir, exist_ok=True)

# 时间序列
time_grid = np.linspace(0, T, N+1)

# 创建单个3D图窗
fig = plt.figure(figsize=(16, 12))  # 调整为适合单个图形的大小
ax = fig.add_subplot(111, projection='3d')  # 创建单个3D子图

# 颜色设置
colors = cm.viridis(np.linspace(0, 1, 11))[::-1]  # 生成从深到浅的颜色渐变

# 对单个参数组合进行模拟
for idx, params in enumerate(param_combinations):
    alpha = params['alpha']
    beta = params['beta']
    label = params['label']
    
    # 初始化路径数组
    A_paths = np.zeros((n_simulations, N+1))
    
    # 初始条件
    A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    
    # 模拟路径
    for i in range(N):
        t = i * dt
        sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
        # 确保A值在[0, pi]范围内，避免数值问题
        A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
        # \Phi 和 \Phi^{-1} 的计算
        Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
        Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

        # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
        dW = np.sqrt(dt) * np.random.randn(n_simulations)
        dA = pi * Phi_prime * sqrt_zt * dW
        
        # 更新A值并确保在[0, pi]范围内
        A_next = A_current + dA
        A_paths[:, i+1] = np.clip(A_next, 0, pi)

    # 选取多个时间节点来绘制KDE - 跳过t=0，从t=0+δ开始
    # 我们跳过第一个时间点(t=0)，因为它是固定的初始值
    time_points = np.linspace(0, N, 11, dtype=int)  # 从索引1开始，选取11个时间点
    
    # 在3D图中绘制KDE曲线
    for i, t in enumerate(time_points):
        # 计算当前时间点的KDE
        kde_data = A_paths[:, t]
        kde = stats.gaussian_kde(kde_data, bw_method=0.5)
        x_vals = np.linspace(0, pi, 200)  # A值的范围
        y_vals = kde(x_vals)  # 密度值
        
        # 当前时间点
        current_time = time_grid[t]
        
        # 在3D图中绘制曲线：x=A值, z=密度, y=时间
        ax.plot(x_vals, np.full_like(x_vals, current_time), y_vals, 
                color=colors[i], linewidth=4, alpha=0.8, label=f't={current_time:.2f}')
    
    # 设置坐标轴标签
    ax.set_xlabel('A(t)', labelpad=28, fontsize=28)
    ax.set_ylabel('Time', labelpad=28, fontsize=28)
    ax.set_zlabel('Density', labelpad=28, fontsize=28)
    ax.set_title(f'3D Time-evolving KDE of Carbon Allowance Prices ({label})', 
                pad=20, fontsize=28, fontweight='bold')
    
    # 设置坐标轴范围
    # ax.set_xlim(0, pi)
    # ax.set_ylim(0, T)
    
    # 调整视角以便更好地观察
    ax.view_init(elev=25, azim=-50)

# 创建时间图例
time_handles = [mlines.Line2D([0], [0], color=colors[i], lw=3) 
               for i in range(len(time_points))]
time_labels = [f't={time_grid[t]:.2f}' for t in time_points]

# 添加全局图例
fig.legend(time_handles, time_labels, 
           loc='lower center', 
           bbox_to_anchor=(0.5, -0.05),
           ncol=6,  # 一行显示6个图例项
           title='Time Points',
           frameon=True, 
           framealpha=0.9,
           fontsize=10)

# 调整布局，为图例留出更多空间
plt.tight_layout(rect=[0, 0.05, 1, 0.95])

# 保存图片
plt.savefig(os.path.join(plots_dir, "3D_Single_Param_Dynamics_of_Carbon_Allowance_Prices.png"), 
           dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats
import seaborn as sns
from matplotlib import cm
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# ================= 参数设置 =================
T = 4          # 终止时间
dt = 0.01           # 时间步长
N = int(T / dt)     # 时间步数量
n_simulations = 10000  # 模拟路径数量

# SDE参数
pi = 100             # pi常数

# 参数组合列表 - 只保留alpha=1.2, beta=0.8
param_combinations = [
    {'alpha': 1.2, 'beta': 0.8, 'label': 'α=1.2, β=0.8'}
]

# 创建保存图表的目录
SAVE_DIR = "./generated/one-period-result-fig"
plots_dir = os.path.join(SAVE_DIR, "carbon-dynamic")
os.makedirs(plots_dir, exist_ok=True)

# 时间序列
time_grid = np.linspace(0, T, N+1)

# 对每个参数组合进行模拟
for idx, params in enumerate(param_combinations):
    alpha = params['alpha']
    beta = params['beta']
    label = params['label']
    
    # 初始化路径数组
    A_paths = np.zeros((n_simulations, N+1))
    
    # 初始条件
    A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    
    # 模拟路径
    for i in range(N):
        t = i * dt
        sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
        # 确保A值在[0, pi]范围内，避免数值问题
        A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
        # \Phi 和 \Phi^{-1} 的计算
        Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
        Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

        # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
        dW = np.sqrt(dt) * np.random.randn(n_simulations)
        dA = pi * Phi_prime * sqrt_zt * dW
        
        # 更新A值并确保在[0, pi]范围内
        A_next = A_current + dA
        A_paths[:, i+1] = np.clip(A_next, 0, pi)

    # 选取四个特定时间点
    time_points = [0, int(N/3), int(2*N/3), N]  # t=0, 1/3T, 2/3T, T
    time_labels = ['t=0', 't=1/3T', 't=2/3T', 't=T']
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # 四种不同颜色
    
    # 创建图形
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # 计算水平偏移量
    max_density = 0
    kde_data = []
    
    for i, t in enumerate(time_points):
        kde = stats.gaussian_kde(A_paths[:, t], bw_method=0.5)
        x_vals = np.linspace(0, pi, 1000)
        y_vals = kde(x_vals)
        kde_data.append((x_vals, y_vals))
        max_density = max(max_density, np.max(y_vals))
    
    # 设置水平偏移参数
    horizontal_shift = 0.15 * pi  # 每个分布之间的水平距离
    vertical_spacing = 0.25 * max_density  # 垂直间距
    
    # 绘制水平偏移的KDE
    for i, (t, color, time_label) in enumerate(zip(time_points, colors, time_labels)):
        x_vals, y_vals = kde_data[i]
        
        # 计算偏移量 - 随时间向右移动
        x_offset = i * horizontal_shift
        y_offset = 0  # 不在垂直方向偏移，保持在同一水平线上
        
        # 绘制KDE曲线
        ax.plot(x_vals + x_offset, y_vals + y_offset, 
                color=color, linewidth=2.5, label=time_label)
        
        # 填充曲线下方区域（半透明）
        ax.fill_between(x_vals + x_offset, y_vals + y_offset, alpha=0.3, color=color)
        
        # 添加时间点标签
        max_y_idx = np.argmax(y_vals)
        ax.text(x_vals[max_y_idx] + x_offset, y_vals[max_y_idx] + y_offset + 0.02, 
                time_label, fontsize=12, ha='center', va='bottom', 
                bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))
        
        # 添加垂直参考线
        ax.axvline(x=x_offset, color=color, linestyle='--', alpha=0.5, linewidth=1)
    
    # 设置图形属性
    ax.set_xlabel('Carbon Allowance Price A(t)', fontsize=14)
    ax.set_ylabel('Probability Density', fontsize=14)
    ax.set_title(f'Time Evolution of Carbon Allowance Price Distribution ({label})', fontsize=16)
    
    # 设置坐标轴范围
    ax.set_xlim(-0.1 * pi, pi + 3 * horizontal_shift)
    ax.set_ylim(0, max_density * 1.2)
    
    # 添加网格
    ax.grid(True, alpha=0.3)
    
    # 添加图例
    ax.legend(loc='upper right', fontsize=12)
    
    # 保存图片
    plt.tight_layout()
    # plt.savefig(os.path.join(plots_dir, f"Time_Evolution_KDE_{alpha}_{beta}.png"), 
            #    dpi=300, bbox_inches='tight')
    plt.show()

    # =================================================================
    # 可选：创建传统堆叠KDE图作为对比
    # =================================================================
    fig2, ax2 = plt.subplots(1, 1, figsize=(10, 8))
    
    for i, (t, color, time_label) in enumerate(zip(time_points, colors, time_labels)):
        sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, label=time_label, ax=ax2, linewidth=2)
    
    ax2.set_xlabel('Carbon Allowance Price A(t)', fontsize=14)
    ax2.set_ylabel('Probability Density', fontsize=14)
    ax2.set_title(f'Traditional KDE Plot ({label})', fontsize=16)
    ax2.legend(fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    # plt.savefig(os.path.join(plots_dir, f"Traditional_KDE_{alpha}_{beta}.png"), 
            #    dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:

# # 参数设置
# T = 4          # 终止时间
# dt = 0.01           # 时间步长
# N = int(T / dt)     # 时间步数量
# n_simulations = 10000  # 模拟路径数量

# # SDE参数
# pi = 100             # pi常数

# beta = 0.2         # 参数beta


# alpha = 0.1# 参数alpha
# # beta = 0.161         # 参数beta
# # alpha = 0.332        # 参数alpha

# SAVE_DIR="./one-period-result-fig"
# # 初始化路径数组
# A_paths = np.zeros((n_simulations, N+1))

# # 初始条件
# A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi

# # 时间序列
# time_grid = np.linspace(0, T, N+1)


# # 模拟路径
# for i in range(N):
#     t = i * dt
#     sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
#     # \Phi 和 \Phi^{-1} 的计算
#     Phi_inv_A = stats.norm.ppf(A_paths[:, i] / pi)  # 计算 \Phi^{-1}(A/pi)
#     Phi_prime = stats.norm.pdf(Phi_inv_A)           # 计算 \Phi'(Phi^{-1}(A/pi))

#     # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
#     dW = np.sqrt(dt) * np.random.randn(n_simulations)
#     dA = pi * Phi_prime * sqrt_zt * dW
#     A_paths[:, i+1] = A_paths[:, i] + dA

# # 分析结果：计算终点的分布
# A_T = A_paths[:, -1]

# # 创建保存图表的目录
# plots_dir = os.path.join(SAVE_DIR, "simulation_plots")
# os.makedirs(plots_dir, exist_ok=True)

# # 选取多个时间节点来绘制KDE
# time_points = np.linspace(0, N, 11, dtype=int)  # 选取10个时间点
# colors = cm.viridis(np.linspace(0, 1, len(time_points)))[::-1]  # 生成从深到浅的颜色渐变

# # =================================================================
# # 修复版本：正确收集图例句柄
# # =================================================================

# # 方法1：使用显式收集图例句柄
# plt.figure(figsize=(28, 8))
# ax1 = plt.subplot(1, 2, 1)

# # 创建图例的句柄和标签
# handles = []
# labels = []

# for i, (t, color) in enumerate(zip(time_points, colors)):
#     # 显式获取线条对象
#     sns_line = sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, 
#                           label=f'Time = {time_grid[t]:.2f}', ax=ax1)
    
#     # 获取最后添加的线对象（更可靠的方法）
#     if ax1.lines:  # 确保有线条存在
#         line = ax1.lines[-1]
#         handles.append(line)
#         labels.append(f'Time = {time_grid[t]:.2f}')

# plt.xlabel('A(t)')
# plt.ylabel('Density')
# plt.title('Time-evolving KDE of $A_t$')

# # 方法2：使用get_legend_handles_labels（更简单）
# if not handles:  # 如果方法1失败，使用方法2
#     handles, labels = ax1.get_legend_handles_labels()

# if handles:  # 确保我们有图例句柄
#     # 将图例分成两部分：一部分在左侧，一部分在右侧
#     split_index = len(handles) // 2  # 将图例分成两半

#     # 左侧图例
#     left_legend = plt.legend(handles[:split_index], labels[:split_index], 
#                              loc='upper left', bbox_to_anchor=(0.01, 0.98),
#                              frameon=True, framealpha=0.9)

#     # 右侧图例（添加到轴中但不覆盖左侧图例）
#     plt.gca().add_artist(left_legend)  # 先添加左侧图例
#     plt.legend(handles[split_index:], labels[split_index:], 
#                loc='upper right', bbox_to_anchor=(0.99, 0.98),
#                frameon=True, framealpha=0.9)
# else:
#     print("Warning: No legend handles found. Adding standard legend.")
#     plt.legend(loc='best')

# # =================================================================
# # 第二个子图：终点的直方图和核密度估计
# # =================================================================
# ax2 = plt.subplot(1, 2, 2)
# plt.hist(A_T, bins=50, density=True, alpha=0.5, color='blue', label='Histogram of $A_T$')
# sns.kdeplot(A_T, color='red', label='KDE of $A_T$', ax=ax2)
# plt.xlabel('$A_T$')
# plt.ylabel('Density')
# plt.title(f'Distribution of $A_T$')
# plt.legend()

# # 调整布局并保存
# plt.tight_layout()
# plt.savefig(os.path.join(plots_dir, "Dynamics of Carbon Allowance Prices.png"), dpi=300, bbox_inches='tight')
# plt.show()


In [ ]:

# # 初始化路径数组
# A_paths = np.zeros((n_simulations, N+1))

# # 初始条件
# A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi

# # 时间序列
# time_grid = np.linspace(0, T, N+1)

# # 模拟路径
# for i in range(N):
#     t = i * dt
#     sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
#     # \Phi 和 \Phi^{-1} 的计算
#     Phi_inv_A = stats.norm.ppf(A_paths[:, i] / pi)  # 计算 \Phi^{-1}(A/pi)
#     Phi_prime = stats.norm.pdf(Phi_inv_A)           # 计算 \Phi'(Phi^{-1}(A/pi))

#     # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
#     dW = np.sqrt(dt) * np.random.randn(n_simulations)
#     dA = pi * Phi_prime * sqrt_zt * dW
#     A_paths[:, i+1] = A_paths[:, i] + dA

# # 分析结果：计算终点的分布
# A_T = A_paths[:, -1]
# # 绘制时间演变的KDE图
# plt.figure(figsize=(28, 8))
# plt.subplot(1, 2, 1)
# # 选取多个时间节点来绘制KDE
# time_points = np.linspace(0, N, 11, dtype=int)  # 选取10个时间点
# colors = cm.viridis(np.linspace(0, 1, len(time_points)))[::-1]  # 生成从深到浅的颜色渐变

# for i, (t, color) in enumerate(zip(time_points, colors)):
#     sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, label=f'Time = {time_grid[t]:.2f}')

# plt.xlabel('A(t)')
# plt.ylabel('Density')
# plt.title('Time-evolving KDE of $A_t$')
# plt.legend(ncol=2)

# # 绘制直方图和核密度估计
# plt.subplot(1, 2, 2)
# plt.hist(A_T, bins=50, density=True, alpha=0.5, color='blue', label='Histogram of $A_T$')
# sns.kdeplot(A_T, color='red', label='KDE of $A_T$')
# plt.xlabel('$A_T$')
# plt.ylabel('Density')
# plt.title(f'Distribution of $A_T$')
# plt.legend(ncol=1)
# plt.show()


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import os
# import scipy.stats as stats
# import seaborn as sns
# from matplotlib import cm

# # ================= 参数设置 =================
# T = 4          # 终止时间
# dt = 0.01           # 时间步长
# N = int(T / dt)     # 时间步数量
# n_simulations = 10000  # 模拟路径数量

# # SDE参数
# pi = 100             # pi常数

# # 参数组合列表
# param_combinations = [
#     {'alpha': 1.2, 'beta': 0.8, 'label': 'α=1.2, β=0.8'},
#     {'alpha': 0.8, 'beta': 0.5, 'label': 'α=1.2, β=0.5'},
#     {'alpha': 0.4, 'beta': 0.2, 'label': 'α=0.4, β=0.2'}
# ]

# # 创建保存图表的目录
# SAVE_DIR = "./generated/multi-param-results"
# plots_dir = os.path.join(SAVE_DIR, "simulation_plots")
# os.makedirs(plots_dir, exist_ok=True)

# # 时间序列
# time_grid = np.linspace(0, T, N+1)

# # 创建1×3的子图
# fig, axes = plt.subplots(1, 3, figsize=(36, 6))

# # 颜色设置
# colors = cm.viridis(np.linspace(0, 1, 11))[::-1]  # 生成从深到浅的颜色渐变

# # 对每个参数组合进行模拟
# for idx, params in enumerate(param_combinations):
#     alpha = params['alpha']
#     beta = params['beta']
#     label = params['label']
    
#     # 初始化路径数组
#     A_paths = np.zeros((n_simulations, N+1))
    
#     # 初始条件
#     A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    
#     # 模拟路径
#     for i in range(N):
#         t = i * dt
#         sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
#         # 确保A值在[0, pi]范围内，避免数值问题
#         A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
#         # \Phi 和 \Phi^{-1} 的计算
#         Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
#         Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

#         # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
#         dW = np.sqrt(dt) * np.random.randn(n_simulations)
#         dA = pi * Phi_prime * sqrt_zt * dW
        
#         # 更新A值并确保在[0, pi]范围内
#         A_next = A_current + dA
#         A_paths[:, i+1] = np.clip(A_next, 0, pi)

    
#     # 选取多个时间节点来绘制KDE
#     time_points = np.linspace(0, N, 11, dtype=int)  # 选取10个时间点
    
#     # 在当前子图中绘制
#     ax = axes[idx]
    
#     # 创建图例的句柄和标签
#     handles = []
#     labels = []
    
#     for i, (t, color) in enumerate(zip(time_points, colors)):
#         # 显式获取线条对象
#         sns_line = sns.kdeplot(A_paths[:, t], bw_adjust=0.5, color=color, 
#                               label=f'Time = {time_grid[t]:.2f}', ax=ax)
        
#         # 获取最后添加的线对象
#         if ax.lines:
#             line = ax.lines[-1]
#             handles.append(line)
#             labels.append(f'Time = {time_grid[t]:.2f}')
    
#     ax.set_xlabel('A(t)')
#     ax.set_ylabel('Density')
#     ax.set_title(f'Time-evolving KDE of $A_t$ ({label})')
    
#     # 添加图例
#     if handles:
#         # 将图例分成两部分：一部分在左侧，一部分在右侧
#         split_index = len(handles) // 2
        
#         # 左侧图例
#         left_legend = ax.legend(handles[:split_index], labels[:split_index], 
#                                loc='upper left', bbox_to_anchor=(0.01, 0.98),
#                                frameon=True, framealpha=0.9)
        
#         # 右侧图例（添加到轴中但不覆盖左侧图例）
#         ax.add_artist(left_legend)  # 先添加左侧图例
#         ax.legend(handles[split_index:], labels[split_index:], 
#                  loc='upper right', bbox_to_anchor=(0.99, 0.98),
#                  frameon=True, framealpha=0.9)
#     else:
#         print(f"Warning: No legend handles found for {label}. Adding standard legend.")
#         ax.legend(loc='best')

# # 调整布局并保存
# plt.tight_layout()
# plt.savefig(os.path.join(plots_dir, "Multi_Param_Dynamics_of_Carbon_Allowance_Prices.png"), 
#            dpi=300, bbox_inches='tight')
# plt.show()

# # =================================================================
# # 第二个图窗：终点的直方图和核密度估计
# # =================================================================
# fig2, axes2 = plt.subplots(1, 3, figsize=(36, 6))

# # 对每个参数组合进行模拟（再次运行，或者可以保存A_T结果）
# for idx, params in enumerate(param_combinations):
#     alpha = params['alpha']
#     beta = params['beta']
#     label = params['label']
    
#     # 初始化路径数组
#     A_paths = np.zeros((n_simulations, N+1))
    
#     # 初始条件
#     A_paths[:, 0] = 0.5 * pi  # 假设初始值 A_0 = 0.5 * pi
    

#     # 模拟路径
#     for i in range(N):
#         t = i * dt
#         sqrt_zt = np.sqrt(beta * (T - t)**-alpha)
        
#         # 确保A值在[0, pi]范围内，避免数值问题
#         A_current = np.clip(A_paths[:, i], 1e-10, pi - 1e-10)
        
#         # \Phi 和 \Phi^{-1} 的计算
#         Phi_inv_A = stats.norm.ppf(A_current / pi)  # 计算 \Phi^{-1}(A/pi)
#         Phi_prime = stats.norm.pdf(Phi_inv_A)       # 计算 \Phi'(Phi^{-1}(A/pi))

#         # dA_t = pi * \Phi'(Phi^{-1}(A/pi)) * sqrt(z_t) * dW_t
#         dW = np.sqrt(dt) * np.random.randn(n_simulations)
#         dA = pi * Phi_prime * sqrt_zt * dW
        
#         # 更新A值并确保在[0, pi]范围内
#         A_next = A_current + dA
#         A_paths[:, i+1] = np.clip(A_next, 0, pi)

#     # 分析结果：计算终点的分布
#     A_T = A_paths[:, -1]
    
#     # 在当前子图中绘制
#     ax = axes2[idx]
    
#     # 绘制直方图和KDE
#     ax.hist(A_T, bins=50, density=True, alpha=0.5, color='blue', label='Histogram of $A_T$')
#     sns.kdeplot(A_T, color='red', label='KDE of $A_T$', ax=ax)
#     ax.set_xlabel('$A_T$')
#     ax.set_ylabel('Density')
#     ax.set_title(f'Distribution of $A_T$ ({label})')
#     ax.legend()

# # 调整布局并保存
# plt.tight_layout()
# plt.savefig(os.path.join(plots_dir, "Multi_Param_Distribution_of_AT.png"), 
#            dpi=300, bbox_inches='tight')
# plt.show()